In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from scipy import stats
from sklearn.metrics import mean_squared_error

print("=======================================================", flush=True)
print(" ENTRENAMIENTO DEL MODELO ARX Y VALIDACIÓN ESTADÍSTICA", flush=True)
print("=======================================================", flush=True)

# 1. CARGA DE DATOS Y GENERACIÓN DE REZAGOS ESPECÍFICOS
print("[1/5] Cargando datos y generando Lags de 8h (PM) y 17h (Radiación)...", flush=True)
df = pd.read_parquet('data/ml_ready/dataset_ozono_predictivo.parquet')

estaciones = df.groupby('Estacion')

df['PM2.5_lag_8'] = estaciones['PM2.5_12h'].shift(8)
df['PM10_lag_8'] = estaciones['PM10_12h'].shift(8)
df['SR_lag_17'] = estaciones['SR'].shift(17)

df_model = df.dropna().reset_index(drop=True)

# 2. DEFINICIÓN DE VARIABLES (X e Y)
print("[2/5] Preparando variables...", flush=True)
y = df_model['O3_8h']

X_cols = ['PM2.5_lag_8', 'PM10_lag_8', 'SR_lag_17', 'TOUT_lag_2', 'NOX_lag_1', 'U_Wind', 'V_Wind']
X = df_model[X_cols]
X = sm.add_constant(X)

# 3. ENTRENAMIENTO DEL MODELO ARX (Con el 100% de los datos)
print("[3/5] Entrenando Modelo ARX (OLS)...", flush=True)
modelo_arx = sm.OLS(y, X).fit()

# 4. EXTRACCIÓN DE MÉTRICAS
print("\n--- MÉTRICAS DEL MODELO ---", flush=True)
print(f"R-cuadrado (R^2): {modelo_arx.rsquared:.4f}", flush=True)
print(f"R-cuadrado Ajustado: {modelo_arx.rsquared_adj:.4f}", flush=True)

predicciones = modelo_arx.predict(X)
rmse = np.sqrt(mean_squared_error(y, predicciones))
print(f"RMSE (Error Cuadrático Medio): {rmse:.4f} ppb", flush=True)

print("\n--- COEFICIENTES BETA (Importancia de Variables) ---", flush=True)
print(modelo_arx.params, flush=True)


# =====================================================================
# 5. PRUEBAS DE SUPUESTOS ESTADÍSTICOS (Exigencia de la Rúbrica)
# =====================================================================
print("\n=======================================================", flush=True)
print(" PRUEBAS DE SUPUESTOS (TEOREMA DE GAUSS-MARKOV)", flush=True)
print("=======================================================", flush=True)

# ---> EL TRUCO PARA QUE NO SE TRABE TU LAPTOP <---
# Tomamos una muestra aleatoria de 50k registros solo para el VIF y BP
X_sample = X.sample(50000, random_state=42)
resid_sample = modelo_arx.resid.loc[X_sample.index]

# A. Multicolinealidad (VIF)
print("\n1. Factor de Inflación de Varianza (VIF) - [Ideal < 5]:", flush=True)
vif_data = pd.DataFrame()
vif_data["Variable"] = X_sample.columns
vif_data["VIF"] = [variance_inflation_factor(X_sample.values, i) for i in range(X_sample.shape[1])]
print(vif_data[vif_data['Variable'] != 'const'], flush=True) 

# B. Autocorrelación de Residuos (Durbin-Watson)
print("\n2. Prueba de Durbin-Watson - [Ideal entre 1.5 y 2.5]:", flush=True)
dw_stat = durbin_watson(modelo_arx.resid)
print(f"Estadístico Durbin-Watson: {dw_stat:.4f}", flush=True)
if 1.5 < dw_stat < 2.5:
    print("✅ Resultado: Ausencia de autocorrelación grave.", flush=True)
else:
    print("⚠️ Resultado: Existe autocorrelación temporal en los residuos (común en series de tiempo, justifica el uso futuro de VAR).", flush=True)

# C. Homocedasticidad (Breusch-Pagan)
print("\n3. Prueba de Breusch-Pagan (Homocedasticidad):", flush=True)
bp_test = het_breuschpagan(resid_sample, X_sample)
bp_pvalue = bp_test[1]
print(f"P-valor: {bp_pvalue:.4e}", flush=True)
if bp_pvalue > 0.05:
    print("✅ Resultado: Residuos homocedásticos (Varianza constante).", flush=True)
else:
    print("⚠️ Resultado: Heterocedasticidad detectada (Los errores varían en magnitud, justifica enfoques dinámicos).", flush=True)

# D. Normalidad de Residuos (Kolmogorov-Smirnov)
print("\n4. Prueba de Kolmogorov-Smirnov (Normalidad de Residuos):", flush=True)
residuos_std = (modelo_arx.resid - np.mean(modelo_arx.resid)) / np.std(modelo_arx.resid)
ks_stat, ks_pvalue = stats.kstest(residuos_std, 'norm')
print(f"Estadístico KS: {ks_stat:.4f}, P-valor: {ks_pvalue:.4e}", flush=True)
if ks_pvalue > 0.05:
    print("✅ Resultado: Los residuos siguen una distribución normal.", flush=True)
else:
    print("⚠️ Resultado: Los residuos no son perfectamente normales (Típico en leyes atmosféricas de cola pesada).", flush=True)

print("\n🚀 EJECUCIÓN FINALIZADA. Copia estos valores para tu LaTeX.", flush=True)

 ENTRENAMIENTO DEL MODELO ARX Y VALIDACIÓN ESTADÍSTICA
[1/5] Cargando datos y generando Lags de 8h (PM) y 17h (Radiación)...
[2/5] Preparando variables...
[3/5] Entrenando Modelo ARX (OLS)...

--- MÉTRICAS DEL MODELO ---
R-cuadrado (R^2): 0.3513
R-cuadrado Ajustado: 0.3512
RMSE (Error Cuadrático Medio): 12.2123 ppb

--- COEFICIENTES BETA (Importancia de Variables) ---
const           7.178703
PM2.5_lag_8    -0.017397
PM10_lag_8      0.051267
SR_lag_17     -14.595595
TOUT_lag_2      0.874180
NOX_lag_1      -0.082197
U_Wind         -0.183550
V_Wind          0.221897
dtype: float64

 PRUEBAS DE SUPUESTOS (TEOREMA DE GAUSS-MARKOV)

1. Factor de Inflación de Varianza (VIF) - [Ideal < 5]:
      Variable       VIF
1  PM2.5_lag_8  1.792624
2   PM10_lag_8  1.816370
3    SR_lag_17  1.045717
4   TOUT_lag_2  1.219939
5    NOX_lag_1  1.157391
6       U_Wind  1.204070
7       V_Wind  1.077026

2. Prueba de Durbin-Watson - [Ideal entre 1.5 y 2.5]:
Estadístico Durbin-Watson: 0.0756
⚠️ Resultado: Exist